(mmm_control_dimensionality)=
# Control Dimensionality and ROAS

How many control variables should an MMM include, and what does the answer cost you?

The instinct is that controls are cheap. They are not the treatment, they are not what you report, and leaving one out risks omitted-variable bias — so the temptation is to throw in everything: holidays, weather, competitor pricing, macro indices, one dummy per promotion. There is a well-founded theoretical worry that this quietly damages the number you actually care about, **total media ROAS**, through the prior alone. This notebook builds the experiment that tests it, and reports both halves of the answer.

The problem is not confounding. We deliberately generate controls that are *independent of media spend*, so they cannot bias ROAS through a back-door path — in the vocabulary of [Cinelli, Forney and Pearl (2024)](https://doi.org/10.1177/00491241221099552) they are **neutral controls**. Anything we observe is therefore attributable to prior geometry alone.

## What goes wrong in the prior

PyMC-Marketing gives each control coefficient an independent `Normal` prior, and each yearly-seasonality coefficient an independent `Laplace` prior, and neither scale depends on how many coefficients there are. Add covariates and the *total* variance the model expects those blocks to explain grows linearly in their count, while the prior on the residual scale stays put. The implied prior on $R^2$ therefore marches towards 1, and the share of prior variance left for media collapses with it — a prior that has decided, before seeing any data, that the model explains everything and media explains none of it.

The headline number arrives earlier than you would expect. With the library defaults and **no control columns at all**, the implied prior $R^2$ is already 0.998 and the share of prior variance left for media is 0.001. Four seasonal coefficients under `Laplace(b=1)` are enough, on a max-scaled target, to exhaust the prior before a single control is added.

The dimension-aware alternative budgets a *fixed* amount of explained variance and splits it across however many coefficients happen to be present. That is the **R2D2** prior of [Zhang, Naughton, Bondell and Reich (2022)](https://doi.org/10.1080/01621459.2020.1825449): put a prior on $R^2$, convert it into a total coefficient-variance budget, and divide that budget over the coefficients with a Dirichlet. Its implied prior $R^2$ is flat in $K$, by construction.

## Seasonality is one of those blocks

`yearly_seasonality=2` is the sort of line nobody thinks twice about, and it is not free either. It adds four covariates — `sin_1`, `sin_2`, `cos_1`, `cos_2` — and in our data exactly one of them does anything: the seasonality in the target *is* `sin_1`, so the remaining three have a true coefficient of zero. They are the same phantom columns as the tail of a control block, arriving through a keyword argument instead of a dataframe.

So seasonality goes **inside** the budget here rather than beside it. Every fit in the notebook carries `yearly_seasonality=2`, and the R2D2 arm is declared with `dims={"control": "control", "fourier": "fourier_mode"}`, which puts one fixed variance allowance across the control coefficients and the Fourier coefficients together. That is both the more usual situation — practitioners reach for seasonality before they reach for their tenth control — and a sharper test of the argument, because three of the four seasonal covariates are known in advance to deserve nothing.

## What that costs in the posterior

<!-- STALE-NUMBERS: every XX below reads off the executed outputs; refresh after re-running. -->

At $K = 24$ controls against $n = 130$ weeks — an entirely ordinary MMM — the reported number moves. Total ROAS comes out at XX under the library defaults and XX under hand-tightened versions of the same priors, against XX under R2D2 and a true 2.008. In absolute error that is XX and XX against XX.

The comparison that isolates *why* is the one between the two independent-prior arms. They differ by a factor of 20 in coefficient scale and land within XX of each other. So this is not a story about picking a better `sigma`; a modeller who tightens the defaults by hand buys XX. What separates R2D2 is that its budget knows how many coefficients it is being divided among.

The trajectory across $K$ is worth stating in full:

| | $K=0$ | $K=5$ | $K=10$ | $K=24$ |
| --- | --- | --- | --- | --- |
| Library default | XX | XX | XX | XX |
| Hand-tightened | XX | XX | XX | XX |
| Split R2D2 | XX | XX | XX | XX |

Bias in total ROAS, truth 2.008. Note that $K = 0$ is not a misspecified model here: seasonality is in every fit, so the $K = 0$ column is an MMM that is missing only the twenty-four persistent controls, and the four seasonal coefficients are the only thing the control prior arms differ over.

The mechanism is not where you would look for it. All three arms agree on how much media *varies* (XX of target variance, against a true 0.150). They disagree on how much media *is*: media's share of mean sales comes out at XX and XX under the independent priors, against XX under R2D2 and a true 0.151, with the intercept taking the other side of the trade. ROAS is a level, not a variance, which is why a diagnostic built on variance shares would show almost nothing while the reported number moves.

Two things this notebook cannot tell you. XX of the 12 fits' 90% intervals cover the truth, so the priors are separated by point accuracy and interval width more than by coverage. And this is one dataset: a single realisation cannot establish the size of the effect, which is what the repetition study at the end is for.

## Relationship to the source paper

This notebook is a media-mix translation of Experiment 4 of [*To select or not to select*](https://arxiv.org/abs/2606.22850) (Section 5.4). The paper studies a randomised experiment with a treatment $z$, a treatment effect $\alpha$, and $p$ neutral covariates, and asks how the posterior for $\alpha$ evolves as $p$ grows under different priors. The mapping:

| Paper | This notebook |
| --- | --- |
| treatment $z$ | media spend, passed through adstock and saturation |
| treatment effect $\alpha$ | **total media ROAS** |
| covariates $X$, drawn iid, independent of $z$ | control variables, persistent but independent of spend |
| $M_{\text{base}}$ (treatment only) | an MMM with `control_columns=None`, still carrying `yearly_seasonality=2` |
| $M_{\text{full}}$ at $p \in \{10, 50, 100\}$ | an MMM fit on the first $K$ controls, $K \in \{5, 10, 24\}$, plus four Fourier nodes throughout |
| $n_{\text{obs}} = 150$ | $n = 130$ weeks (two and a half years of weekly data) |

Two details of the mapping are worth flagging up front. First, the base model is the *punchline*, not a footnote: Table 4 of the paper reports that the Normal full model is **worse** than the base model (RMSE 0.30 vs 0.27, coverage 0.87 vs 0.92), so "add the controls, they cannot hurt" is exactly the belief under test. Second, of the paper's three prior specifications we implement **Normal** and **split** (R2D2 on the controls and the Fourier nodes, media priors untouched). The joint R2D2, which shrinks the media coefficients too and which the paper shows to be badly biased, is out of scope.

Two deliberate departures. First, we stay well *inside* the paper's dimension range rather than pushing past it. The paper runs up to $p/n \approx 0.67$; our largest fit is 24 controls and 4 Fourier nodes against $n = 130$, so $p/n \approx 0.22$. That is the boring, everyday MMM — two and a half years of weekly data, yearly seasonality, and a couple of dozen candidate controls — and it is the more useful test, because it is easy to make any prior look bad at $p \approx n$ where the model is barely identified and the sampler is in trouble anyway. The question a practitioner needs answered is whether these priors are already doing damage at dimensions that look entirely reasonable.

Second, our controls are **persistent** rather than iid. Real candidate controls are smooth series — prices, weather indices, macro aggregates — and persistence is what turns a control block from an orthogonal basis into a set of weakly identified directions: mean pairwise $|\text{corr}|$ comes out at 0.150 here against 0.072 for a white-noise block of the same shape, so the likelihood no longer pins each coefficient down on its own and the prior has something to arbitrate. Persistence also buys extra capacity to mimic media, at these dimensions: the diagnostics below show the persistent block reaching a higher $R^2$ against the true media contribution than a white-noise block of the same size, because persistence correlates a column with any other slow-moving signal, media included, not only with its neighbours in the block.

## Prepare Notebook

In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import seaborn as sns
import xarray as xr
from pymc_extras.prior import Prior
from pytensor.xtensor.type import as_xtensor

from pymc_marketing.constants import DAYS_IN_YEAR
from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.fourier import YearlyFourier
from pymc_marketing.mmm.mmm import MMM
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation
from pymc_marketing.r2d2 import R2D2

warnings.filterwarnings("ignore", category=FutureWarning)

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

The experiment is driven by a handful of constants. `N_DATES` sets the shape of the problem; `K_MAX` is the size of the decaying-importance ladder, of which slot 0 is seasonality and the rest, `N_CONTROLS = K_MAX - 1`, are the candidate controls; `K_GRID` is the sequence of nested control subsets we fit; `FOURIER_ORDER` is the `yearly_seasonality` every model carries, whatever $K$ is.

`FOURIER_NODES` is read off `YearlyFourier` rather than written out by hand, because the order matters twice over: it is the order the MMM lays out `gamma_fourier` in, and therefore the order the R2D2 Dirichlet slices the budget in. Note that it is all the sines and then all the cosines, so at order 2 it reads `sin_1, sin_2, cos_1, cos_2`.

In [ ]:
SEED = 42

N_DATES = 130  # two and a half years of weekly data
N_CHANNELS = 5
L_MAX = 8  # adstock window, in weeks
K_MAX = 25  # size of the decaying-importance ladder: slot 0 is seasonality,
# the remaining K_MAX - 1 are the candidate controls
N_CONTROLS = K_MAX - 1
CHOL_ETA = 10.0  # LKJ concentration for the spend correlation matrix
CONTROL_PHI = 0.85  # weekly persistence of the control series
FOURIER_ORDER = 2  # `yearly_seasonality`, in every fit

K_GRID = [0, 5, 10, N_CONTROLS]

CHANNELS = [f"x{i + 1}" for i in range(N_CHANNELS)]
CONTROLS = [f"c{k + 1}" for k in range(N_CONTROLS)]
DATES = pd.date_range("2021-01-04", freq="W-MON", periods=N_DATES)

# All the sines, then all the cosines — the order the MMM builds the Fourier
# basis in, and so the order the R2D2 Dirichlet slices it in.
FOURIER_NODES = YearlyFourier(n_order=FOURIER_ORDER, prefix="fourier_mode").nodes
PHANTOM_NODES = [node for node in FOURIER_NODES if node != "sin_1"]

## The data-generating process

We need a dataset where the true total ROAS is known exactly and identical for every $K$, so that any movement in the posterior is attributable to the coefficient priors and nothing else. We build it in four pieces: spend, controls, seasonality, and the target.

### Media spend

Channel spend comes from an LKJ-Cholesky generative model, the same construction used in the parameter recovery notebook, {ref}`mmm_data_generator`. It gives realistically correlated channel spend with per-channel trends, and `softplus` enforces non-negativity.

One deviation from that notebook: the per-channel scale prior is `Gamma(6, 2)` rather than `Exponential(1/3)`. Both have mean 3, but the Exponential regularly draws values near zero, which produces a channel whose spend barely moves across the whole period. Because the MMM divides each channel by its own maximum before saturating, such a channel sits permanently at the flat top of its saturation curve and contributes an almost constant term — one that is not separable from the intercept. The Gamma keeps every channel's spend on a usable dynamic range.

In [ ]:
def draw_spend(rng: np.random.Generator) -> np.ndarray:
    """Draw one realisation of correlated, trending, non-negative channel spend."""
    coords = {"channel": CHANNELS, "date": np.arange(N_DATES)}
    t = np.arange(N_DATES) / N_DATES

    with pm.Model(coords=coords) as spend_model:
        t_data = pm.Data("t", t, dims=("date",))
        L, _, _ = pm.LKJCholeskyCov(
            "L",
            n=N_CHANNELS,
            eta=CHOL_ETA,
            sd_dist=pm.Gamma.dist(alpha=6, beta=2),
        )
        a = pm.Normal("a", mu=0, sigma=1, dims="channel")
        b = pm.Normal("b", mu=0, sigma=1, dims="channel")
        mu = pm.Deterministic("mu", a + b * t_data[..., None], dims=("date", "channel"))
        x_raw = pm.MvNormal("x_raw", mu=mu, chol=L, dims=("date", "channel"))
        pm.Deterministic("x", pt.softplus(x_raw), dims=("date", "channel"))

    return pm.draw(spend_model.x, draws=1, random_seed=rng)

### The true media contribution

We call `geometric_adstock` and `logistic_saturation` from `pymc_marketing.mmm.transformers` directly, at fixed true `alpha` and `lam`, so the truth matches the model's functional form *exactly* rather than approximately. Two settings have to line up with what the `MMM` class does internally, or the truth will silently disagree with the model:

- the transformations are applied to **max-scaled** spend, `x / x.max()`, because `MMM` max-scales each channel before the forward pass;
- `normalize=True`, because that is the default on the `GeometricAdstock` wrapper even though the underlying `geometric_adstock` function defaults to `False`.

`beta` enters `LogisticSaturation` as a pure multiplicative scale, so evaluating the shape once at `beta = 1` lets us solve the variance budget below in closed form. This is why we call the transformers directly instead of routing the target through `pm.do` and `sample_prior_predictive`, which would need a two-pass loop to hit a variance target.

In [ ]:
def media_shape(
    spend_scaled: np.ndarray, alpha: np.ndarray, lam: np.ndarray
) -> np.ndarray:
    """Adstock then saturation at unit beta, matching the MMM's forward pass."""
    adstocked = geometric_adstock(
        as_xtensor(spend_scaled, dims=("date", "channel")),
        alpha=as_xtensor(alpha, dims=("channel",)),
        l_max=L_MAX,
        dim="date",
        normalize=True,
    )
    saturated = logistic_saturation(adstocked, lam=as_xtensor(lam, dims=("channel",)))
    return saturated.transpose("date", "channel").values.eval()

### Controls, seasonality, variance budget, and the target

Underneath the controls and the seasonal term is one **decaying-importance ladder** of $K_{\max}$ slots, standardised and independent of spend. Standardising matters here because, unlike the target and the channels, `MMM` does **not** scale controls — so `gamma_control` lives in scaled-target-per-raw-control units, and standardised columns are what keep that interpretable. It is not a formality either: before rescaling, the persistent columns range over roughly 0.7 to 1.3 in standard deviation, so without it the ladder below would be scrambled by scale noise and a shared variance budget would be comparing unlike things.

**Slot 0 of that ladder is seasonality, not a control.** It is a yearly sine wave, and it is the one slot in the ladder whose contribution the model sees through its own `yearly_seasonality` term instead of a control column — which is how an MMM practitioner would actually treat seasonality. Concretely, the raw regressor is $\sin(2\pi\, \text{dayofyear} / 365.25)$, which is *bit-for-bit* the `sin_1` column `YearlyFourier` generates, so the true `gamma_fourier` is exact rather than approximate and the remaining three nodes at order 2 are exactly zero. Slots 1 through $K_{\max}-1$ stay controls, exactly as before.

Because standardising a column both rescales *and* recentres it, and the model's raw Fourier basis does neither, converting slot 0's standardised weight into a coefficient on the raw basis leaves behind a constant — the sine's own mean over a non-integer number of years, which is small but not zero at $n = 130$ weeks. That constant is folded into the intercept rather than dropped, which is what the model would do with it anyway: a Fourier term has no way to claim a constant for itself. Every other true quantity in `simulate` — the coefficients, the variance shares, $R^2$ — is unaffected, since adding a constant to a time series never changes its variance.

Two things follow from routing slot 0 through `yearly_seasonality`. $K = 0$ is no longer a misspecified model: it sees the season perfectly well, and is merely missing the persistent controls. And **the seasonal/control split is read off the ladder, not chosen**: there is still one `delta_media` knob fixing media's share against everything else, and slot 0's share of that "everything else" is whatever the decay formula below assigns it — reported as `delta_seasonal` further down, a derived quantity rather than an input.

Slots 1 through $K_{\max}-1$ are **AR(1) with $\phi = 0.85$**, not white noise, with the innovation scale chosen to keep the stationary variance at 1. This is the departure that makes the experiment interesting: persistent columns are mutually correlated in a finite sample — with $\phi = 0.85$ the effective sample size per column is around 11, not 130 — so the control block is internally collinear rather than an orthogonal basis, and the likelihood no longer pins each coefficient down on its own. That is measured in the cell after next, together with a second and more consequential kind of overlap: how much of a *smooth* signal, media's or seasonality's, a block of smooth columns can reproduce.

```{warning}
**Spend does not depend on the season here.** Real marketers spend into high season, which makes seasonality a genuine *confounder* of media and one of the main reasons to include it. Our spend is generated without reference to either the controls or the calendar, so every covariate in this notebook — the seasonal term included — remains **neutral** in the Cinelli–Forney–Pearl sense. That is what isolates the prior-geometry effect we want to measure, but it also means nothing here speaks to the confounding case. If your spend follows the season, you are in a different problem, and dropping seasonality biases ROAS for reasons that have nothing to do with this notebook.
```

The true coefficients **decay** down the whole ladder, $\gamma_k \propto (1 - k/K_{\max})^2$ for $k = 0, \ldots, K_{\max}-1$, seasonality included at $k = 0$. This is a deliberate departure from the paper, which gives every covariate the same coefficient (`beta <- array(0.1, c(p, 1))`). The decaying version is the situation an analyst is actually in: you order candidate covariates by how much you believe in them — and seasonality is usually the covariate you believe in *most* — so the first few genuinely matter and the tail is close to noise. The square makes that ordering steep enough to bite — the last control's coefficient is not merely smaller than the first's, it is negligible — so each step up the $K$ grid buys less signal than the last, while under an independent prior it costs just as much variance budget as the first.

The rest of the DGP is a variance budget with three knobs:

- **`media_contribution_share`** — the fraction of total sales driven by media. Since $\mathbb{E}[y] = \text{baseline} + \mathbb{E}[\text{media}]$ and the controls, seasonality and noise are all mean-zero (up to the small intercept offset above), targeting a share $s$ means $\mathbb{E}[\text{media}] = s \cdot \text{baseline} / (1 - s)$, which fixes the overall scale of `saturation_beta`. Default 0.15.
- **`delta_media`** — media's share of *explained* variance, the analogue of equation (34) in the paper. The media level above already fixes media's variance, so this knob fixes the size of the pot the whole ladder — seasonality and controls together — divides up. Default 0.2.
- **`true_r2`** — the fraction of the target's variance that media, seasonality and controls jointly explain, which pins the residual scale (the paper's footnote 15). Default 0.75. Together with `delta_media` this fixes the quantity the R2D2 prior is actually a prior on — the ladder's variance against the residual, media excluded from both — at $0.8 / (0.8 + 1/3) \approx 0.706$. A less noisy target would push that above 0.9, which is far enough into the corner of the unit interval that no honest Beta prior can be centred near it.

Finally we divide the target by its own maximum. This is not cosmetic: `MMM` max-scales the target, so normalising here makes the model's internal scale exactly 1 and lets us report every true parameter in the units the model actually works in.

In [ ]:
def simulate(
    seed: int,
    *,
    media_contribution_share: float = 0.15,
    true_r2: float = 0.75,
    delta_media: float = 0.2,
    baseline: float = 1.0,
    spend_to_revenue: float = 0.075,
) -> tuple[pd.DataFrame, dict]:
    """Generate one dataset together with its exact ground truth.

    The decaying-importance ladder is untouched from the version of this
    notebook that fed slot 0 to the model as a control: one `delta_media`
    knob still splits explained variance into media versus "everything
    else", and the same `importance` decay still weights all `K_MAX` slots,
    slot 0 included. The only change is which regressor slot 0's weight ends
    up multiplying: instead of the standardised column becoming `c1`, it
    becomes the coefficient on the model's own `yearly_seasonality` term.
    That split is read off the ladder, not chosen separately.
    """
    rng = np.random.default_rng(seed)

    spend_raw = draw_spend(rng)
    spend_scaled = spend_raw / spend_raw.max(axis=0)

    alpha_true = rng.beta(1, 3, size=N_CHANNELS)
    lam_true = rng.gamma(shape=3, scale=1, size=N_CHANNELS)
    shape = media_shape(spend_scaled, alpha_true, lam_true)

    # Relative betas that equalise each channel's mean contribution, so that no
    # single channel dominates the media term.
    beta_rel = 1.0 / shape.mean(axis=0)
    shape_unit = shape @ beta_rel

    # Slot 0 is a yearly sine wave; slots 1..K_MAX-1 are AR(1), not white
    # noise: real candidate controls are persistent series, and persistence
    # is what lets a control block partially reproduce a smooth media signal
    # in a finite sample. The innovation scale keeps the stationary variance
    # at 1. All K_MAX columns are standardised together, exactly as when
    # slot 0 was itself fed to the model as a control.
    columns = np.empty((N_DATES, K_MAX))
    columns[:, 0] = np.sin(2 * np.pi * DATES.dayofyear.to_numpy() / DAYS_IN_YEAR)
    columns[0, 1:] = rng.normal(size=K_MAX - 1)
    innovations = rng.normal(
        scale=np.sqrt(1 - CONTROL_PHI**2), size=(N_DATES, K_MAX - 1)
    )
    for t in range(1, N_DATES):
        columns[t, 1:] = CONTROL_PHI * columns[t - 1, 1:] + innovations[t]
    seasonal_raw = columns[:, 0].copy()  # pre-standardisation, for the model's basis
    columns = (columns - columns.mean(axis=0)) / columns.std(axis=0)

    media_mean = media_contribution_share * baseline / (1 - media_contribution_share)
    beta_true = beta_rel * media_mean / shape_unit.mean()
    media = shape @ beta_true

    var_media = media.var()
    var_budget = var_media * (1 - delta_media) / delta_media  # seasonal + controls
    var_explained = var_media + var_budget

    # Importance decays down the whole ladder, slot 0 (seasonality) included,
    # so the leading slots carry real signal and the trailing ones are all
    # but noise. This is the one computation the two DGP versions share
    # byte-for-byte: slot 0's weight is whatever this decay says it is, not
    # a separately chosen belief about seasonality's share.
    importance = (1.0 - np.arange(K_MAX) / K_MAX) ** 2
    ladder_signal = columns @ importance
    gamma_true = importance * np.sqrt(var_budget / ladder_signal.var())

    # Standardising slot 0 also recentres it, which the model's raw Fourier
    # basis does not do. Convert: gamma_true[0] * standardised(t) =
    # gamma_fourier_true * seasonal_raw(t) + intercept_offset, an identity
    # for every t. The offset is a constant, so it is folded into the
    # intercept rather than lost — the model would attribute a sine's mean
    # over a non-integer number of years to its intercept too.
    raw_std = seasonal_raw.std()
    gamma_fourier_true = gamma_true[0] / raw_std
    intercept_offset = -gamma_true[0] * seasonal_raw.mean() / raw_std
    seasonal = gamma_fourier_true * seasonal_raw
    control = columns[:, 1:] @ gamma_true[1:]

    sigma_true = np.sqrt(var_explained * (1 - true_r2) / true_r2)
    noise = rng.normal(scale=sigma_true, size=N_DATES)

    y_raw = baseline + intercept_offset + media + seasonal + control + noise
    scale = y_raw.max()
    y = y_raw / scale

    spend_level = spend_to_revenue * y.sum() / spend_raw.sum()
    spend = spend_raw * spend_level

    # Only the first sine is real. The other three nodes the model carries at
    # `yearly_seasonality=2` have a true coefficient of exactly zero.
    gamma_fourier_vec = np.zeros(len(FOURIER_NODES))
    gamma_fourier_vec[FOURIER_NODES.index("sin_1")] = gamma_fourier_true / scale

    budget = seasonal + control  # everything the R2D2 decomposition covers
    truth = {
        # True parameters, expressed in the model's internal (scaled-target) units.
        "intercept": (baseline + intercept_offset) / scale,
        "adstock_alpha": alpha_true,
        "saturation_lam": lam_true,
        "saturation_beta": beta_true / scale,
        "gamma_control": gamma_true[1:] / scale,
        "gamma_fourier": gamma_fourier_vec,
        "y_sigma": sigma_true / scale,
        # Realised summaries of this particular dataset.
        "media_contribution": media / scale,
        "control_contribution": control / scale,
        "seasonal_contribution": seasonal / scale,
        "total_roas": (media / scale).sum() / spend.sum(),
        "media_contribution_share": media.sum() / y_raw.sum(),
        "delta_media": var_media / var_explained,
        # Not a knob: the share of the shared budget slot 0's ladder weight
        # happens to carry, read off gamma_true rather than chosen.
        "delta_seasonal": gamma_true[0] ** 2 / np.sum(gamma_true**2),
        "media_var_share": var_media / (var_explained + sigma_true**2),
        "seasonal_var_share": seasonal.var() / (var_explained + sigma_true**2),
        "r2": (media + seasonal + control).var() / y_raw.var(),
        # The quantity the R2D2 prior is actually a prior on: the control block
        # and the seasonal term together against the residual, with media
        # excluded from every term.
        "budget_r2": budget.var() / (budget.var() + sigma_true**2),
        "target_scale": scale,
    }

    frame = {"date": DATES}
    frame |= {channel: spend[:, i] for i, channel in enumerate(CHANNELS)}
    frame |= {
        control_name: columns[:, 1 + k] for k, control_name in enumerate(CONTROLS)
    }
    frame["y"] = y

    return pd.DataFrame(frame), truth

We draw the dataset **once** and hold it fixed for every model below. `y` is therefore byte-identical across all fits, and the nested subsets $K \in$ `K_GRID` all come from the same pool of `N_CONTROLS` columns.

In [ ]:
data, truth = simulate(SEED)

pd.Series(
    {
        "true total ROAS": truth["total_roas"],
        "media share of sales": truth["media_contribution_share"],
        "intercept share of sales": truth["intercept"] / data["y"].mean(),
        "media share of explained variance": truth["delta_media"],
        "seasonal share of the combined budget (read off the ladder)": truth[
            "delta_seasonal"
        ],
        "media share of target variance": truth["media_var_share"],
        "seasonality share of target variance": truth["seasonal_var_share"],
        "true R-squared": truth["r2"],
        "true budget-block R-squared (against residual)": truth["budget_r2"],
        "true residual sigma (scaled target)": truth["y_sigma"],
        "true gamma, first control (scaled target)": truth["gamma_control"][0],
        "true gamma, last control (scaled target)": truth["gamma_control"][-1],
        "true gamma_fourier, sin_1 (scaled target)": truth["gamma_fourier"][
            FOURIER_NODES.index("sin_1")
        ],
        "target scale": truth["target_scale"],
    }
).round(4)

Ground-truth total ROAS is the total true media contribution divided by total spend — a single number, identical for every $K$ because the data never change.

The controls are independent of spend by construction, but with $n = 130$ the *sample* correlations are not exactly zero, and it is finite-sample overlap rather than population dependence that lets a neutral control move ROAS. Three measurements, in increasing order of how much they matter, and then the same question asked of the seasonal block.

**Correlation with spend.** The direct route: a control that happens to track a channel competes with it for attribution. These stay small.

**Correlation within the block.** The persistent columns are mutually correlated even though they are independent by construction, and this is what turns the control block from an orthogonal basis into a set of weakly identified directions. Compare against a white-noise block of the same shape to see how much of it is persistence rather than sample size.

**How much of media the block can mimic.** The sharpest version of the question: regress the *true* media contribution on the first $K$ controls and read off the $R^2$. This is the capacity the control block has to explain away media, and it is the quantity a variance budget is limiting. Again with a white-noise block for contrast, and here persistence does buy something: at $K = 24$ the persistent block reaches 0.199 where white noise reaches only 0.107. Smoothness costs us identification within the block, as above, and it also buys the block extra capacity to mimic any other slow-moving signal — media included.

**And the Fourier block.** Worth asking of seasonality too, since every fit carries it. The four nodes reach 0.056 on media between them, and the three with a true coefficient of zero reach 0.040 on their own — capacity that exists purely because somebody typed `yearly_seasonality=2` instead of `1`.

The last line is the one that justifies putting both blocks in one budget: **the twenty-four controls can reproduce 0.875 of the true seasonal signal.** Twenty-four persistent columns over two and a half years trace a yearly cycle very well, so the control block and the seasonal block are not two separate concerns competing with media independently — they are largely fighting over the same variance. A budget that covers one and not the other is only half a budget.

In [ ]:
spend_matrix = data[CHANNELS].to_numpy()
control_matrix = data[CONTROLS].to_numpy()
cross_corr = np.corrcoef(spend_matrix.T, control_matrix.T)[:N_CHANNELS, N_CHANNELS:]

white_noise = np.random.default_rng(SEED).normal(size=(N_DATES, N_CONTROLS))

# The design matrix the model's Fourier term regresses on, built by hand so it
# can be measured here.  Identical to `generate_fourier_modes`, node for node.
periods = DATES.dayofyear.to_numpy() / DAYS_IN_YEAR
fourier_matrix = np.column_stack(
    [np.sin(2 * np.pi * (order + 1) * periods) for order in range(FOURIER_ORDER)]
    + [np.cos(2 * np.pi * (order + 1) * periods) for order in range(FOURIER_ORDER)]
)
phantom_matrix = fourier_matrix[:, [FOURIER_NODES.index(n) for n in PHANTOM_NODES]]


def off_diagonal(block: np.ndarray) -> np.ndarray:
    """Absolute pairwise correlations between the columns of a block."""
    corr = np.corrcoef(block.T)
    return np.abs(corr[~np.eye(block.shape[1], dtype=bool)])


def block_r2(target: np.ndarray, block: np.ndarray) -> float:
    """Share of `target` variance a control block plus intercept can reproduce."""
    design = np.column_stack([np.ones(len(target)), block])
    coefficients, *_ = np.linalg.lstsq(design, target, rcond=None)
    residual = target - design @ coefficients
    return 1.0 - residual.var() / target.var()


print(f"max |corr(spend, control)| : {np.abs(cross_corr).max():.3f}")
print(f"mean |corr(spend, control)|: {np.abs(cross_corr).mean():.3f}")

print("\nwithin-block correlation, against a white-noise block of the same size")
for label, block in [("persistent", control_matrix), ("white noise", white_noise)]:
    pairwise = off_diagonal(block)
    print(f"  {label:11s} max {pairwise.max():.3f}   mean {pairwise.mean():.3f}")

print("\nshare of the true media contribution the first K controls can mimic")
for k in [k for k in K_GRID if k > 0]:
    persistent = block_r2(truth["media_contribution"], control_matrix[:, :k])
    iid = block_r2(truth["media_contribution"], white_noise[:, :k])
    print(f"  K={k:3d}  persistent: {persistent:.3f}   white noise: {iid:.3f}")

print("\nthe same question asked of the Fourier block, which every fit carries")
print(
    f"  all {len(FOURIER_NODES)} nodes:                       "
    f"{block_r2(truth['media_contribution'], fourier_matrix):.3f}"
)
print(
    f"  the {len(PHANTOM_NODES)} zero-coefficient nodes alone:  "
    f"{block_r2(truth['media_contribution'], phantom_matrix):.3f}"
)
print(
    f"  and of the seasonal signal, by the {N_CONTROLS} controls: "
    f"{block_r2(truth['seasonal_contribution'], control_matrix):.3f}"
)

### Looking at the data

In [ ]:
fig, axes = plt.subplots(nrows=3, figsize=(12, 9), sharex=True, layout="constrained")

sns.lineplot(x="date", y="y", data=data, color="black", ax=axes[0])
axes[0].plot(
    DATES,
    truth["intercept"] + truth["seasonal_contribution"],
    color="C1",
    linestyle="--",
    label="intercept + true seasonality",
)
axes[0].set(title="Target (max-scaled sales)", ylabel="y")
axes[0].legend(loc="upper left", fontsize=8)

for channel in CHANNELS:
    sns.lineplot(x="date", y=channel, data=data, ax=axes[1], label=channel, alpha=0.8)
axes[1].set(title="Channel spend", ylabel="spend")
axes[1].legend(ncol=5, loc="upper left", fontsize=8)

for control_name in CONTROLS[:5]:
    sns.lineplot(
        x="date", y=control_name, data=data, ax=axes[2], alpha=0.6, linewidth=0.9
    )
axes[2].set(
    title=(
        f"First 5 of {N_CONTROLS} controls "
        f"(AR(1) with phi={CONTROL_PHI}; all independent of spend)"
    ),
    ylabel="control value",
    xlabel="date",
)

fig.suptitle("The fixed dataset, shared by every model below", fontsize=14)
plt.show()

Because importance decays, the $K$ grid is not a grid of equal steps in information. The cumulative share of control variance below makes that concrete: the first handful of columns carry most of the control signal and the tail adds very little. Under an independent prior, though, every column in that tail costs exactly as much prior variance as the informative ones at the head did.

In [ ]:
gamma_true = truth["gamma_control"]
cumulative_share = np.cumsum(gamma_true**2) / np.sum(gamma_true**2)

fig, axes = plt.subplots(ncols=2, figsize=(13, 4), layout="constrained")

axes[0].plot(np.arange(1, len(gamma_true) + 1), gamma_true, color="C0")
axes[0].set(
    xlabel="control index k",
    ylabel=r"true $\gamma_k$",
    title="True control coefficients decay down the block",
)

axes[1].plot(np.arange(1, len(gamma_true) + 1), cumulative_share, color="C0")
axes[1].scatter(
    [k for k in K_GRID if k > 0],
    [cumulative_share[k - 1] for k in K_GRID if k > 0],
    color="C1",
    zorder=3,
    label="K in the experiment grid",
)
axes[1].set(
    xlabel="number of controls K",
    ylabel="cumulative share",
    title="Share of control variance captured by the first K controls",
    ylim=(0, 1.05),
)
axes[1].legend(fontsize=9)

plt.show()

## The coefficient priors

Three arms, and only two ideas: independent priors at the library default scales, the same priors tightened by hand, and R2D2. The two independent arms are there to separate *scale* from *dimension* — if tightening the scales were enough, they would differ.

All three touch only `model_config["gamma_control"]` and `model_config["gamma_fourier"]`, which is the whole point: the media priors are untouched **by construction**, which is what makes the third one a *split* prior rather than a joint one. No `MuEffect` subclass and no changes to the `MMM` class are needed.

Here is the code path they feed into. `gamma_control` is created, multiplied by the control data, summed over controls, and added to `mu`; `gamma_fourier` arrives the same way, via `YearlyFourier.apply`:

```python
if self.control_columns is not None and len(self.control_columns) > 0:
    gamma_control = self.model_config["gamma_control"].create_variable(
        name="gamma_control", xdist=True
    )
    control_data_ = pmd.Data("control_data", self.xarray_dataset._control)
    control_contribution = pmd.Deterministic(
        "control_contribution", control_data_ * gamma_control
    )
    mu_var += control_contribution.sum(dim="control")
```

Because both blocks are configured this way, the three arms differ **at every $K$, including $K = 0$**: with no controls there is still a four-coefficient seasonal block to put a prior on. The $K = 0$ column is therefore three separate fits, not one shared baseline.

### 1. Independent priors

The library defaults are `Prior("Normal", mu=0, sigma=2, dims="control")` and `Prior("Laplace", mu=0, b=1, dims="fourier_mode")`. Note what is missing from both: the scale does not depend on how many coefficients there are. The prior variance the two blocks together expect to explain is $K\sigma^2 + 2 M b^2$ for $M$ Fourier nodes (a `Laplace(b)` has variance $2b^2$), growing without bound as covariates are added.

Comparing R2D2 against those defaults alone would be both unfair and uninformative. On a max-scaled target `sigma=2` is so diffuse that the implied prior $R^2$ is pinned at 1 almost immediately, and *any* prior looks good next to one that has already declared the model explains everything. So we give the independent priors the benefit of a modeller who has noticed that the defaults are too wide for a max-scaled target and has tightened them by hand.

`SIGMA_TIGHT = 0.1` is the value you land on by reasoning about **scale**: the target is max-scaled into roughly $[0.4, 1]$, a decent model should leave a residual of order 0.1, so an individual coefficient of order 0.1 is plausible and one of order 1 is not. It is applied to both blocks — `sigma` on the controls and `b` on the Fourier nodes — and we use the same 0.1 for the residual scale prior (`SIGMA_REF`), so the coefficient priors and the expected residual are stated on a common footing.

This is a real improvement in the prior — it drops the implied prior $R^2$ at $K = 24$ from 1.000 to 0.982, and at $K = 0$ from 0.998 to 0.866 — and it is the honest comparison to make, because tightening a too-wide default is exactly what a careful practitioner does. But note what it is *not*: reasoning about scale is not reasoning about **dimension**. `SIGMA_TIGHT` is still a constant, so the total budget still grows with the number of coefficients. Keeping both arms means that everything we attribute to dimension-awareness later has to survive a 20-fold change in coefficient scale, which turns out to be a much sterner test of the story than comparing against the defaults alone.

In [ ]:
SIGMA_REF = 0.1  # prior scale for the residual sigma, on the scaled-target scale
SIGMA_TIGHT = 0.1  # hand-tightened scale, for both coefficient blocks

n_fourier = len(FOURIER_NODES)

print(f"control sigma: {SIGMA_TIGHT} (library default is 2)")
print(f"Fourier b:     {SIGMA_TIGHT} (library default is 1)")
print(
    f"implied block budget at K={N_CONTROLS} plus {n_fourier} Fourier nodes: "
    f"{N_CONTROLS * SIGMA_TIGHT**2:.3f} + {n_fourier * 2 * SIGMA_TIGHT**2:.3f} = "
    f"{N_CONTROLS * SIGMA_TIGHT**2 + n_fourier * 2 * SIGMA_TIGHT**2:.3f}"
)
print(
    "  (Laplace(b) has variance 2b^2, so a Fourier node costs twice what a "
    "control does at the same scale)"
)

### 3. Split R2D2

`pymc_marketing.r2d2.R2D2` ships exactly this prior, so what follows is the model it implements, not a build log. The construction, following `r2_normal.stan` from the paper's reference implementation:

$$
R^2 \sim \text{Beta}\!\left(\bar{r}\,\kappa,\ (1-\bar{r})\,\kappa\right), \qquad
\tau^2 = \frac{R^2}{1 - R^2}, \qquad
\psi \sim \text{Dirichlet}(\mathbf{1}_K), \qquad
\gamma_k = z_k \sqrt{\sigma^2 \tau^2 \psi_k},\quad z_k \sim \mathcal{N}(0, 1),
$$

where $\sigma^2 \tau^2$ is a **fixed** variance budget and the Dirichlet weights $\psi$ split it among however many coefficients there are — the total does not grow with $K$. Here $K$ counts *both* blocks: the control coefficients and the four Fourier coefficients draw from one flat Dirichlet with $K + 4$ cells, so the seasonal block's expected share of the budget is $4 / (K + 4)$, from all of it at $K = 0$ down to about a seventh at $K = 24$.

`R2D2` parametrises $\sigma$ slightly differently from the derivation above: instead of taking the residual scale as an external input, it puts a prior on `total_sigma` — the combined scale of the component(s) named in `dims` plus the residual — and *derives* both the model-side budget and the residual scale from it and $R^2$:

$$
\sigma_{\text{model}} = \sqrt{R^2}\ \sigma_{\text{total}}, \qquad
\sigma_{\text{resid}} = \sqrt{1 - R^2}\ \sigma_{\text{total}}.
$$

`dims` maps component name to model dim, and does not have to cover every term in the model — here it covers the controls and the Fourier nodes, so media stays outside the decomposition entirely, exactly as in the derivation above:

```python
R2D2(
    r2=Prior("Beta", alpha=r2_alpha, beta=r2_beta),
    total_sigma=Prior("HalfNormal", sigma=TOTAL_SIGMA_SCALE),
    dims={"control": "control", "fourier": "fourier_mode"},
)
```

(`r2_alpha`, `r2_beta` and `TOTAL_SIGMA_SCALE` are derived below, not borrowed from `SIGMA_REF`.)

The decomposition has to be built per $K$ rather than once at module scope, which is what `make_r2d2` is for: at $K = 0$ the model has no `control` dim, so naming one in `dims` would fail. The `"fourier"` entry is always there.

`split("control")`, `split("fourier")` and `error_sigma` are all lazy references: whichever is used first inside a model builds the whole decomposition (Beta, Dirichlet, offsets) once, and the others look it up — the same "one variable, several consumers" trick this notebook used to hand-roll via a `SharedScale` helper, now built into the class. Since the Dirichlet has to know its own length the moment it is built, every component's dim must exist in the model before the first `create_variable` call; that `fourier_mode` is registered early enough for this to work is [a fix](https://github.com/pymc-labs/pymc-marketing/pull/2929) this notebook prompted.

It is also why the likelihood changes alongside the coefficients: `model_config_for` swaps in `Prior("Normal", sigma=r2d2.error_sigma, dims="date")` only for the Split R2D2 arm, since only that arm's residual scale is derived this way; the two independent arms keep a plain `HalfNormal(SIGMA_REF)`.

#### An elicited budget, not a borrowed one

`total_sigma` needs its own scale, and the path of least resistance is to reach for `SIGMA_REF` again — it is already sitting there, stated in the same units. But nothing connects `SIGMA_REF` to what this parameter means: it was chosen for the Normal arm's coefficient scale, not for "the combined scale of the control block plus the residual," and reusing it here would be exactly the kind of borrowed-without-justification number this notebook argues against everywhere else. The same problem applies to `r2_mean`: stating a bare 0.6 and calling it "a statement an analyst is in a position to make" begs the question of what informed it.

The fix is to elicit two dimensionless statements a modeller can actually reason about, in the units they already think in, *before* touching the data:

- $R^2_{\text{total}} = 0.7$ — "I expect a reasonable model, media and controls together, to explain about 70% of the target's variance."
- $\delta_{\text{media}} = 0.3$ — "Of what gets explained, I'd guess media accounts for a bit less than half; the rest is seasonality and other controls."

Together with $\operatorname{Var}(y)$ — the target's own sample variance, which needs no belief at all — these pin down everything R2D2 needs:

$$
\operatorname{Var}(\text{media}) = \delta_{\text{media}}\, R^2_{\text{total}}\, \operatorname{Var}(y), \qquad
\sigma_{\text{total}} = \sqrt{\operatorname{Var}(y) - \operatorname{Var}(\text{media})}, \qquad
\bar r = \frac{R^2_{\text{total}}(1-\delta_{\text{media}})}{1 - \delta_{\text{media}}\, R^2_{\text{total}}}.
$$

The first line carves media's believed share out of the target's variance before R2D2 ever sees a covariate. The second hands R2D2 whatever is left over — controls, seasonality and residual, which is exactly what `total_sigma` means once `dims` covers both blocks. The third restates $\bar r$ in the same terms as before: how much of that leftover budget the two blocks should take from the residual. Notice $\bar r$ does not depend on $\operatorname{Var}(y)$ at all — only the *absolute* scale does; the *split* is a pure ratio of beliefs.

Note how well the second elicited statement now fits what the model does. "The rest is seasonality and other controls" is not a turn of phrase here: seasonality and the controls are precisely the two components inside `dims`, so the belief and the decomposition line up term for term.

This gives $\sigma_{\text{total}} \approx 0.097$ and $\bar r \approx 0.620$, i.e. $\text{Beta}(13.6, 8.4)$ with 95% of its mass in $[0.41, 0.81]$ — not asserted but derived: they follow from two sentences a modeller can defend rather than from reusing a neighbouring constant. The truth for this dataset is 0.706, comfortably inside that range but nowhere near its centre, so the prior is informative without being handed the answer.

This is also where the "don't quietly underestimate media" property comes from. Because $\operatorname{Var}(\text{media})$ is subtracted out *first*, no amount of correlation between these blocks and media — the "spare capacity that points at media" mechanism this notebook documents later — can inflate the budget past what the modeller committed to before $K$ grows. The two independent arms carry no such reservation.

More generally, this is where the honesty of a shrinkage prior lives. A budget prior only shrinks while it is more confident than the data; make it vague and it will simply widen to accommodate whatever the likelihood prefers, which is the correct behaviour and also no help at all when the likelihood is the thing you are worried about.

In [ ]:
# Two statements a modeller can defend before seeing the data, in place of
# borrowing SIGMA_REF and asserting r2_mean.  See the markdown above for the
# derivation; Var(y) is the one quantity that needs the data, and it is just
# the target's own sample variance.
R2_TOTAL_BELIEF = 0.7  # "a decent model explains about 70% of the variance"
DELTA_MEDIA_BELIEF = 0.3  # "media is probably a bit less than half of that"
R2_PRECISION = 22.0

var_y = float(data["y"].var())
var_media_belief = DELTA_MEDIA_BELIEF * R2_TOTAL_BELIEF * var_y
TOTAL_SIGMA_SCALE = float(np.sqrt(var_y - var_media_belief))
R2_MEAN = (
    R2_TOTAL_BELIEF
    * (1 - DELTA_MEDIA_BELIEF)
    / (1 - DELTA_MEDIA_BELIEF * R2_TOTAL_BELIEF)
)

print(f"Var(y) = {var_y:.4f} (observed, no belief required)")
print(f"Var(media), implied by belief = {var_media_belief:.4f}")
print(
    f"total_sigma scale = sqrt(Var(y) - Var(media)) = {TOTAL_SIGMA_SCALE:.3f} "
    f"(borrowing {SIGMA_TIGHT} from the hand-tightened arm would carry no such "
    "reasoning)"
)
print(f"r2_mean, implied by belief = {R2_MEAN:.3f}")

r2_alpha, r2_beta = R2_MEAN * R2_PRECISION, (1 - R2_MEAN) * R2_PRECISION
r2_draws = Prior("Beta", alpha=r2_alpha, beta=r2_beta).sample_prior(
    draws=20_000, random_seed=SEED
)["variable"]
lower, upper = (float(r2_draws.quantile(q)) for q in (0.025, 0.975))
print(f"Beta({r2_alpha:.1f}, {r2_beta:.1f}) on the budget block's R^2")
print(f"  median {float(r2_draws.median()):.2f}, 95% [{lower:.2f}, {upper:.2f}]")
print(f"  truth for this dataset: {truth['budget_r2']:.2f}")


def make_r2d2(k: int) -> R2D2:
    """One decomposition, covering the control block and the Fourier block.

    `dims` maps component name to model dim.  At `k = 0` the model has no
    `control` dim at all, so there the budget covers the Fourier nodes alone.
    """
    dims = {"control": "control"} if k > 0 else {}
    dims["fourier"] = "fourier_mode"

    return R2D2(
        r2=Prior("Beta", alpha=r2_alpha, beta=r2_beta),
        total_sigma=Prior("HalfNormal", sigma=TOTAL_SIGMA_SCALE),
        dims=dims,
    )


def _residual_sigma(dataset: xr.Dataset) -> xr.DataArray:
    """Residual sigma, however the active likelihood config derives it.

    Under the Normal control priors this is the named ``y_sigma`` variable.
    Under Split R2D2 it is never its own named variable — it is a
    deterministic function of ``r2d2_r2`` and ``r2d2_total_sigma`` — so it has
    to be reconstructed from those two instead.
    """
    if "y_sigma" in dataset:
        return dataset["y_sigma"]
    return (1 - dataset["r2d2_r2"]) ** 0.5 * dataset["r2d2_total_sigma"]

A quick check that it builds and samples, and that the budget it implies does not grow with $K$ — which is the entire claim. There is no likelihood in this throwaway model, so `error_sigma` is never itself a named variable; `_residual_sigma` reconstructs it from the sampled `r2d2_r2` and `r2d2_total_sigma` instead. Both blocks are created here, so the budget being measured is the one the fits actually use.

The natural thing to report is the budget *relative to* the residual variance, $\sum_k \gamma_k^2 / \sigma^2 = \tau^2 \sum_k \psi_k z_k^2$, since that is the dimensionless quantity the $R^2$ prior actually pins down. Its expectation is $\mathbb{E}[\tau^2] = \alpha / (\beta - 1) = 13.6 / 7.4 = 1.84$, which is what the mean column should reproduce.

Two features of the numbers below are worth anticipating so they are not mistaken for the thing we are testing. The ratio is right-skewed, so its median (1.39 to 1.63) sits below its mean, which holds at 1.82 to 1.85 exactly as predicted. And the implied budget $R^2$ climbs mildly with $K$, from 0.581 at $K = 5$ to 0.619 at $K = 100$, approaching the $\text{Beta}(13.6, 8.4)$ median of 0.62 from below: $\sum_k \psi_k z_k^2$ has mean 1 for every $K$ but only *concentrates* on 1 as $K$ grows, so at small $K$ the realised budget is usually smaller than the intended one. Neither is growth in the budget itself. Contrast the independent priors, whose block budget grows linearly with the coefficient count.

The last column is the part seasonality adds to this story. The Fourier block's realised share of the budget is 0.449, 0.139 and 0.039 at $K$ of 5, 25 and 100, against a flat-Dirichlet expectation of $4/(K+4)$ = 0.444, 0.138 and 0.038 — the mechanism working exactly as advertised. It is also the thing to keep an eye on: a seasonal signal of fixed strength is being asked to live on an allowance that shrinks as controls arrive. The likelihood is free to push the weights back, and whether it does is what the fits below show.

In [ ]:
for k in [5, 25, 100]:
    coords = {
        "control": [f"c{i + 1}" for i in range(k)],
        "fourier_mode": FOURIER_NODES,
    }
    with pm.Model(coords=coords):
        r2d2 = make_r2d2(k)
        r2d2.split("control").create_variable("gamma_control")
        r2d2.split("fourier").create_variable("gamma_fourier")
        draws = pm.sample_prior_predictive(draws=2_000, random_seed=SEED).prior

    y_sigma = _residual_sigma(draws)
    control_budget = (draws["gamma_control"] ** 2).sum("control")
    fourier_budget = (draws["gamma_fourier"] ** 2).sum("fourier_mode")
    budget = control_budget + fourier_budget
    ratio = budget / y_sigma**2
    implied_r2 = budget / (budget + y_sigma**2)
    print(
        f"K={k:4d}  budget/sigma^2: mean={float(ratio.mean()):6.3f} "
        f"median={float(ratio.median()):5.3f}   "
        f"implied budget R^2 median={float(implied_r2.median()):.3f}   "
        f"Fourier share of budget={float((fourier_budget / budget).mean()):.3f} "
        f"(Dirichlet expectation {len(FOURIER_NODES) / (k + len(FOURIER_NODES)):.3f})"
    )

### Two honest caveats

**"$R^2$" here means the budget block against the residual, not the model's $R^2$.** Because `dims` names only the controls and the Fourier nodes, the Beta governs $\operatorname{Var}(\text{control} + \text{seasonality}) / (\operatorname{Var}(\text{control} + \text{seasonality}) + \sigma_{\text{resid}}^2)$, with media excluded from both numerator and denominator. So `r2_mean` ≈ 0.62 is not a claim that the model explains 62% of the target's variance; it is a claim, restated from $R^2_{\text{total}}$ and $\delta_{\text{media}}$, about how much of what media leaves behind the two blocks should be allowed to take. Confusing the two would make the number look far too pessimistic for an MMM. A prior on the model's actual $R^2$ is the *joint* version, which shrinks media too, and which the paper finds considerably worse.

**R2D2 assumes standardised covariates** (footnote 7 of the paper), and here one block obeys and one does not. The controls are standardised by construction, which matters more than in a plain regression because `MMM` does not scale controls itself. The Fourier basis is not standardised: each node has standard deviation $1/\sqrt 2$, so its coefficients need to be $\sqrt 2$ larger to buy the same variance, and a budget shared as though every covariate had unit variance is therefore slightly stingy towards seasonality. It is a fixed factor on one block rather than anything that grows with $K$, so it does not touch the dimension argument, but with controls on wildly different scales a single shared budget stops being meaningful at all — standardise them before using this prior.

## Model configuration

Everything except the two coefficient blocks (and, for R2D2, the residual scale that comes with them) is held identical across every fit. The priors are stated on the scaled-target scale, where the target lies in roughly $[0.4, 1]$: this is worth being explicit about, because the library defaults (`intercept` and `gamma_control` at `sigma=2`, `gamma_fourier` at `Laplace(b=1)`, likelihood `sigma` at `HalfNormal(2)`) are extremely diffuse relative to a max-scaled target, and that diffuseness is a large part of the story.

```{note}
**The media priors are the distributions the truth was drawn from.** `Beta(1, 3)` on `adstock_alpha` and `Gamma(3, 1)` on `saturation_lam` are both the library defaults and the DGP's own draws, so the media block is favourably specified. That is deliberate: it removes media prior misspecification as a competing explanation for anything we observe downstream, leaving the control and seasonality priors as the only moving parts.

It is a real assumption, and a previous run of this experiment measured what relaxing it costs: refitting everything with `Beta(2, 2)` and `Gamma(2, 0.4)` instead — wider, and centred away from the truth — roughly tripled every ROAS error, inflated every interval by about 60%, and reintroduced divergences, without changing the ranking of the three arms. That check has not been repeated against the current data-generating process.
```

In [ ]:
BASE_CONFIG = {
    "intercept": Prior("Normal", mu=0.5, sigma=0.5),
    "adstock_alpha": Prior("Beta", alpha=1, beta=3, dims="channel"),
    "saturation_lam": Prior("Gamma", alpha=3, beta=1, dims="channel"),
    "saturation_beta": Prior("HalfNormal", sigma=0.15, dims="channel"),
    "likelihood": Prior(
        "Normal", sigma=Prior("HalfNormal", sigma=SIGMA_REF), dims="date"
    ),
}

ARM_NAMES = ["Library default", "Hand-tightened", "Split R2D2"]
ARM_COLORS = dict(zip(ARM_NAMES, ["C3", "C1", "C0"], strict=True))


def model_config_for(k: int, arm: str) -> dict:
    """Model config for one (K, arm) cell.

    Only the two coefficient blocks differ between arms — plus, for Split R2D2,
    the likelihood's residual scale, which falls out of the same decomposition.
    """
    config = dict(BASE_CONFIG)

    if arm == "Split R2D2":
        r2d2 = make_r2d2(k)
        config["gamma_fourier"] = r2d2.split("fourier")
        config["likelihood"] = Prior("Normal", sigma=r2d2.error_sigma, dims="date")
        if k > 0:
            config["gamma_control"] = r2d2.split("control")
        return config

    scale = 2.0 if arm == "Library default" else SIGMA_TIGHT
    b = 1.0 if arm == "Library default" else SIGMA_TIGHT
    config["gamma_fourier"] = Prior("Laplace", mu=0, b=b, dims="fourier_mode")
    if k > 0:
        config["gamma_control"] = Prior("Normal", mu=0, sigma=scale, dims="control")
    return config


SAMPLER_CONFIG = {
    "draws": 1_000,
    "tune": 1_000,
    "chains": 4,
    "target_accept": 0.95,
    "nuts_sampler": "nutpie",
}

`target_accept=0.95` is applied to *every* fit, independent priors and R2D2 alike, which keeps the comparison clean. R2D2 needs it more: $\tau^2 = R^2/(1-R^2)$ is a funnel, and deriving both the coefficient budget and the likelihood's residual scale from the same `r2`/`total_sigma` pair couples that funnel to the residual scale.

The informative Beta helps here as a side effect. $\tau^2$ has a finite mean only for $\beta > 1$ and a finite variance only for $\beta > 2$, so the paper's $\text{Beta}(1, 2)$ sits right at the edge of having no variance at all, while $\text{Beta}(13.6, 8.4)$ is comfortably inside. The diagnostics table later reports divergences and $\hat R$ per fit.

Two things about the grid. `K = 0` must pass `control_columns=None`, not `[]`: the `MMM` constructor enforces `min_length=1` on that field. And every fit passes `yearly_seasonality=FOURIER_ORDER`, so the seasonal block is present at $K = 0$ too — which is why all three arms are fit there rather than sharing one baseline.

In [ ]:
def build_mmm(k: int, arm: str) -> MMM:
    """Build an MMM on the first `k` controls under the named prior arm."""
    return MMM(
        date_column="date",
        channel_columns=CHANNELS,
        control_columns=CONTROLS[:k] if k > 0 else None,
        target_column="y",
        adstock=GeometricAdstock(l_max=L_MAX),
        saturation=LogisticSaturation(),
        yearly_seasonality=FOURIER_ORDER,
        model_config=model_config_for(k, arm),
        sampler_config=SAMPLER_CONFIG,
    )

## The mechanism: implied prior $R^2$ against $K$

Before running any MCMC we can see the problem directly, from prior draws alone. This is the analogue of the left panel of Figure 4 in the paper, and it is by far the cheapest diagnostic in this notebook.

For each prior draw we decompose the variance of the linear predictor over time into a media part, a control part and a seasonal part, and add the residual variance:

$$
R^2_{\text{media}} = \frac{\operatorname{Var}_t(\text{media})}{\operatorname{Var}_t(\text{media}) + \operatorname{Var}_t(\text{control}) + \operatorname{Var}_t(\text{seasonality}) + \sigma^2},
\qquad
R^2_{\text{control}} = \frac{\operatorname{Var}_t(\text{control})}{\cdots},
\qquad
R^2_{\text{seasonal}} = \frac{\operatorname{Var}_t(\text{seasonality})}{\cdots},
$$

so the three shares add to the implied prior $R^2$. Everything comes from `sample_prior_predictive`; no sampler is involved.

In [ ]:
def prior_variance_shares(
    k: int, arm: str, draws: int = 2_000
) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    """Media, control and seasonal shares of implied prior variance, per draw."""
    mmm = build_mmm(k, arm)
    mmm.build_model(data.drop(columns=["y"]), data["y"])

    # No var_names filter: Split R2D2 doesn't expose a named `y_sigma` (see
    # `_residual_sigma`), so we sample everything and reconstruct it below.
    with mmm.model:
        prior = pm.sample_prior_predictive(draws=draws, random_seed=SEED).prior

    var_media = prior["channel_contribution"].sum("channel").var("date")
    var_control = (
        prior["control_contribution"].sum("control").var("date")
        if k > 0
        else xr.zeros_like(var_media)
    )
    var_seasonal = prior["yearly_seasonality_contribution"].var("date")
    total = var_media + var_control + var_seasonal + _residual_sigma(prior) ** 2

    return var_media / total, var_control / total, var_seasonal / total

In [ ]:
prior_records = []
for arm in ARM_NAMES:
    for k in K_GRID:
        media, control, seasonal = prior_variance_shares(k, arm)
        r2 = media + control + seasonal
        prior_records.append(
            {
                "arm": arm,
                "K": k,
                "r2_median": float(r2.median()),
                "r2_q05": float(r2.quantile(0.05)),
                "r2_q95": float(r2.quantile(0.95)),
                "media_share": float(media.median()),
                "control_share": float(control.median()),
                "seasonal_share": float(seasonal.median()),
            }
        )

prior_r2 = pd.DataFrame(prior_records)
prior_r2.round(3)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(13, 5.5), sharex=True, layout="constrained")
colors = ARM_COLORS

for label in ARM_NAMES:
    subset = prior_r2.query("arm == @label")
    axes[0].plot(
        subset["K"], subset["r2_median"], marker="o", color=colors[label], label=label
    )
    axes[0].fill_between(
        subset["K"],
        subset["r2_q05"],
        subset["r2_q95"],
        color=colors[label],
        alpha=0.12,
    )

axes[0].axhline(1.0, color="black", linestyle=":", linewidth=1)
axes[0].set(
    xlabel="number of controls K",
    ylabel="implied prior $R^2$",
    title="Implied prior $R^2$ (median, 90% band)",
    ylim=(0, 1.05),
)
axes[0].legend(loc="lower right", fontsize=9)

for label in ARM_NAMES:
    subset = prior_r2.query("arm == @label")
    axes[1].plot(
        subset["K"],
        subset["media_share"],
        marker="o",
        color=colors[label],
        label=label,
    )

axes[1].set(
    xlabel="number of controls K",
    ylabel="median media share of prior variance",
    title="How much prior variance is left for media",
    ylim=(0, 0.8),
)
axes[1].legend(loc="upper right", fontsize=9)

fig.suptitle(
    "The independent priors spend their whole variance budget on the "
    "coefficient blocks as K grows",
    fontsize=14,
)
plt.show()

The three arms do not share a starting point, and that is the first thing to take from the table. At $K = 0$ the implied prior $R^2$ is already 0.998, 0.866 and 0.756. There are no control coefficients yet, so the entire difference comes from four seasonal ones — `Laplace(b=1)`, `Laplace(b=0.1)` and a Dirichlet slice of a fixed budget. **Under the library defaults, seasonality alone exhausts the prior**: the seasonal block takes 0.996 of the implied prior variance and media is left with 0.001. Nobody adding yearly seasonality to an MMM thinks they are making a claim about total explained variance, and this is what the claim looks like.

Adding controls from there does what the arithmetic says. Under the **library defaults** the implied prior $R^2$ is numerically 1 from $K = 5$ on. **Tightening by hand helps but does not fix it**: 0.866 at $K = 0$, 0.942 at $K = 5$, 0.963 at $K = 10$, 0.982 at $K = 24$, climbing monotonically as the coefficient count grows. **R2D2 is flat** — 0.753, 0.785, 0.787, 0.792 across the whole grid. The budget is fixed, so splitting it more finely does not change the total.

The right panel tells the same story from media's point of view, and it is the more alarming one. The share of prior variance left for media runs 0.082, 0.034, 0.022, 0.011 under the tightened arm — a factor of eight lost to columns that are, by construction, incapable of confounding anything — and 0.001 or less throughout under the defaults. Under R2D2 it goes 0.526, 0.462, 0.435, 0.425: it gives ground, because the controls are real and have to be paid for out of a fixed pot, but it stays within a factor of 1.2 of where it started. A prior that has decided in advance that media explains essentially nothing is a prior that starts out unwilling to attribute revenue to media.

Watch what R2D2 does *between* its two blocks, too. The seasonal share falls from 0.139 at $K = 0$ to 0.013 at $K = 24$ while the control share rises from 0 to 0.264 — one allowance, redistributed, rather than two allowances accumulating. That is the same mechanism seen from the inside, and it is why the seasonal coefficients are worth watching in the posterior: their prior allowance shrinks as controls arrive even though the signal they are fitting does not.

## The experiment

Now the posterior. Nested subsets $K \in$ `K_GRID` of the same pool of controls, so `y` is byte-identical across every fit, and `yearly_seasonality=2` throughout. Every $K$ is fit three times, once per arm — including $K = 0$, since the seasonal block gives the arms something to disagree about even with no controls. Everything else — media priors, seeds, sampler settings — is held identical.

Total ROAS per fit comes from the counterfactual machinery rather than from a contribution deterministic, so it is the same quantity a practitioner would report.

In [ ]:
def total_roas(mmm: MMM) -> np.ndarray:
    """Posterior draws of total media ROAS over the whole fitted period."""
    incremental = mmm.incrementality.compute_incremental_contribution(
        frequency="all_time"
    )
    spend = float(mmm.data.get_channel_spend().sum())
    return (incremental.sum("channel") / spend).values.ravel()


def variance_shares(mmm: MMM, k: int) -> dict:
    """Posterior variance shares and implied posterior R^2 for a fitted model."""
    posterior = mmm.idata.posterior
    var_media = posterior["channel_contribution"].sum("channel").var("date")
    var_control = (
        posterior["control_contribution"].sum("control").var("date")
        if k > 0
        else xr.zeros_like(var_media)
    )
    var_seasonal = posterior["yearly_seasonality_contribution"].var("date")
    total = var_media + var_control + var_seasonal + _residual_sigma(posterior) ** 2

    return {
        "media_share": float((var_media / total).mean()),
        "control_share": float((var_control / total).mean()),
        "seasonal_share": float((var_seasonal / total).mean()),
        "posterior_r2": float(
            ((var_media + var_control + var_seasonal) / total).mean()
        ),
    }


def level_shares(mmm: MMM, mean_target: float) -> dict:
    """Mean contribution of the media and intercept blocks, as a share of mean sales.

    Variance shares say how much each block *moves*; ROAS depends on how much
    each block *is*.  The two come apart when a control block soaks up level.
    """
    posterior = mmm.idata.posterior
    media = posterior["channel_contribution"].sum("channel").mean("date")

    return {
        "media_level_share": float(media.mean()) / mean_target,
        "intercept_level_share": float(posterior["intercept_contribution"].mean())
        / mean_target,
    }


def fit_for_k(
    dataset: pd.DataFrame, k: int, arm: str, seed: int
) -> tuple[np.ndarray, dict]:
    """Fit one (K, arm) cell and return ROAS draws plus fit diagnostics."""
    mmm = build_mmm(k, arm)
    mmm.fit(
        dataset.drop(columns=["y"]),
        dataset["y"],
        random_seed=seed,
        progressbar=False,
    )

    # No var_names filter: which sigma variable exists depends on the prior
    # (see `_residual_sigma`), so summarise everything and take the max r_hat.
    diagnostics = az.summary(mmm.idata.posterior)
    posterior = mmm.idata.posterior
    gamma_fourier = posterior["gamma_fourier"]
    info = {
        "K": k,
        "arm": arm,
        "divergences": int(mmm.idata.sample_stats["diverging"].sum()),
        "max_r_hat": float(diagnostics["r_hat"].max()),
        "posterior_sigma": float(_residual_sigma(posterior).mean()),
        # Posterior spread of a single control coefficient, and the R2D2 budget
        # the data end up asking for.  Both feed the shrinkage table below.
        "gamma_sd": float(posterior["gamma_control"].std(["chain", "draw"]).mean())
        if k > 0
        else np.nan,
        "r2_posterior": float(posterior["r2d2_r2"].mean())
        if "r2d2_r2" in posterior
        else np.nan,
        # The seasonal coefficient the data support, and the root-mean-square of
        # the three whose true value is exactly zero.
        "gamma_sin_1": float(gamma_fourier.sel(fourier_mode="sin_1").mean()),
        "phantom_rms": float(
            (gamma_fourier.sel(fourier_mode=PHANTOM_NODES) ** 2).mean() ** 0.5
        ),
    }
    info |= variance_shares(mmm, k)
    info |= level_shares(mmm, float(dataset["y"].mean()))

    return total_roas(mmm), info

The ground-truth check below is worth pausing on. It confirms that our analytic truth is *exactly* the quantity the MMM would report, by plugging the true parameters into the model graph with `pm.do` and reading off its own total media contribution. If the adstock `normalize` flag or the channel scaling were misaligned, this is where it would show up.

In [ ]:
check_mmm = build_mmm(0, ARM_NAMES[0])
check_mmm.build_model(data.drop(columns=["y"]), data["y"])
fixed = pm.do(
    check_mmm.model,
    {
        "intercept_contribution": truth["intercept"],
        "adstock_alpha": truth["adstock_alpha"],
        "saturation_lam": truth["saturation_lam"],
        "saturation_beta": truth["saturation_beta"],
    },
)
model_total = float(pm.draw(fixed["total_media_contribution_original_scale"]))

model_roas = model_total / spend_matrix.sum()

print(f"model-implied total ROAS at true parameters: {model_roas:.6f}")
print(f"analytic ground-truth total ROAS          : {truth['total_roas']:.6f}")

In [ ]:
roas_draws: dict[tuple[int, str], np.ndarray] = {}
fit_records = []

for k in K_GRID:
    for arm in ARM_NAMES:
        draws, info = fit_for_k(data, k, arm, seed=SEED)
        roas_draws[k, arm] = draws
        fit_records.append(info)

fits = pd.DataFrame(fit_records)
fits.round(4)

Every cell of the grid is its own fit, $K = 0$ included, so the tables and figures below are a straight three-by-four. `by_arm` pulls out one arm's series across $K$, and is used by every table and figure from here on.

In [ ]:
def roas_summary(draws: np.ndarray) -> dict:
    """Posterior summary of ROAS against the known truth."""
    q05, median, q95 = np.quantile(draws, [0.05, 0.5, 0.95])
    return {
        "q05": q05,
        "median": median,
        "q95": q95,
        "interval_length": q95 - q05,
        "covers_truth": bool(q05 <= truth["total_roas"] <= q95),
        "bias": float(draws.mean() - truth["total_roas"]),
        "squared_error": float((draws.mean() - truth["total_roas"]) ** 2),
    }


def by_arm(frame: pd.DataFrame, arm: str) -> pd.DataFrame:
    """One row per K for a single arm."""
    return frame[frame["arm"] == arm].sort_values("K")


def roas_matrix(column: str) -> pd.DataFrame:
    """Pivot one ROAS summary column into a K-by-arm table."""
    return pd.concat(
        [
            by_arm(roas_table, name).set_index("K")[column].rename(name)
            for name in ARM_NAMES
        ],
        axis=1,
    )


roas_table = pd.DataFrame(
    [
        {"K": k, "arm": arm} | roas_summary(draws)
        for (k, arm), draws in roas_draws.items()
    ]
)
roas_table.round(4)

### The headline figure: total ROAS across $K$

This is the analogue of Figure 9 in the paper. One posterior density per $K$, one column per prior, with the true ROAS and the $K = 0$ baseline marked.

In [ ]:
def series_for(arm: str) -> list[tuple[int, np.ndarray]]:
    """ROAS draws by K for one arm."""
    return [(k, roas_draws[k, arm]) for k in K_GRID]


fig, axes = plt.subplots(
    ncols=len(ARM_NAMES),
    figsize=(16, 5.5),
    sharex=True,
    sharey=True,
    layout="constrained",
)
palette = sns.color_palette("viridis", n_colors=len(K_GRID))

for ax, arm in zip(axes, ARM_NAMES, strict=True):
    for (k, draws), color in zip(series_for(arm), palette, strict=True):
        sns.kdeplot(
            x=draws, ax=ax, color=color, linewidth=2, label=f"K = {k}", clip=(0, None)
        )
    ax.axvline(
        truth["total_roas"], color="black", linestyle="--", linewidth=2, label="truth"
    )
    baseline_q05, baseline_q95 = np.quantile(roas_draws[0, arm], [0.05, 0.95])
    ax.axvspan(
        baseline_q05,
        baseline_q95,
        color="gray",
        alpha=0.15,
        label="K = 0 baseline, 90%",
    )
    ax.set(xlabel="total ROAS", title=f"{arm}", xlim=(0, 7))

axes[0].set_ylabel("posterior density")
axes[-1].legend(loc="upper right", fontsize=9)
fig.suptitle("Total ROAS posterior as controls are added, by prior arm", fontsize=14)
plt.show()

<!-- STALE-NUMBERS: every XX below reads off the executed outputs; refresh after re-running. -->
At $K = 24$ the medians are XX, XX and XX against a true 2.008. Whether the two independent arms are again indistinguishable from each other, despite the 20-fold difference in the coefficient scale that produced them, is the first thing to check: that is the comparison that separates *scale* from *dimension*.

Read the panels left to right in $K$ rather than across in arm, and the shape of the result appears. The $K = 0$ column is a fair baseline now — seasonality is in the model, so nothing is being blamed on a cycle the fit cannot see — and it under- or over-attributes to media purely because the persistent controls are missing and their signal has nowhere to go but the residual and the media/intercept split. At $K = 0$ the medians are XX, XX and XX, with the residual scale at XX against a true 0.0583.

The comparison against $K = 0$ is worth making precise. Twenty-four controls take the library default's absolute error from XX to XX; the same twenty-four controls, run through R2D2 instead, take it from XX to XX. The controls are doing the same job in both fits; the prior decides how much of that work reaches the reported number.

### Why the priors matter here

Two conditions have to hold for a coefficient prior to reach the reported number, and both are met at $K = 24$ in a way they would not be with iid controls.

**The likelihood no longer pins each coefficient on its own.** With standardised controls, $\sum_t c_{kt}^2 = n = 130$, so the naive likelihood-only standard deviation for a coefficient is $\sigma/\sqrt{n} =$ XX. But the persistent columns are mutually correlated — mean $|\text{corr}|$ of 0.150 against 0.072 for white noise, and a maximum of 0.668 against 0.240 — and once you account for that the mean likelihood-only standard deviation across the 24 coefficients is XX, a variance inflation factor of XX. The observed posterior standard deviation is XX under the independent priors; if it sits close to the likelihood-only value, those priors are doing almost nothing. Under R2D2 it is XX.

**The blocks have spare capacity that points at media.** Regressing the true media contribution on the first $K$ controls gives an $R^2$ of 0.091 at $K = 10$ and 0.199 at $K = 24$. The trailing columns, whose true coefficients are nearly zero, are not harmless: collectively they can reproduce about a fifth of the media signal. The seasonal block adds 0.056 of its own, 0.040 of which comes from the three nodes with a true coefficient of zero. And the two blocks overlap heavily with *each other* — the controls reproduce 0.875 of the true seasonal signal — so they are not two independent claims on media's variance but one largely shared one. A budget prior limits exactly that, by capping what the blocks as a whole are allowed to be, and it does so without needing to know which columns are the culprits.

### Where the damage shows up

The natural guess is that the control block steals variance from media, and previously it was wrong: all three arms put media's share of target variance at almost the same value and all three overshot the truth by about the same margin. Check whether that still holds here — the numbers are XX, XX and XX against a true 0.150.

The disagreement to look for is in the **level**. Media's share of mean sales comes out at XX and XX under the independent priors, against XX under R2D2 and a true 0.151. Those are the ROAS errors, restated: ROAS is total media contribution over total spend, so a level error is the whole story and a variance error is beside the point. The intercept takes the other side of the trade, at XX and XX against a true 0.842. Neither the controls nor the seasonal term take much level themselves, since both are mean-zero by construction — what those blocks do is loosen the media/intercept split rather than compete for level directly.

This is the practical warning in the notebook. A coefficient block that is overfitting shows up faintly in variance decompositions, faintly in $R^2$, not at all in the sampler diagnostics — and at full strength in the one number the model exists to produce.

The arithmetic behind the first of the two conditions above, priced out from the fits rather than asserted. Comparing the two likelihood-only standard deviations shows what the persistent columns cost in identification, and comparing those against the observed posterior spread shows which priors are contributing anything.

In [ ]:
largest_k = max(K_GRID)
at_largest_k = fits.query("K == @largest_k").set_index("arm")
sigma_hat = float(at_largest_k["posterior_sigma"].mean())

design = np.column_stack([np.ones(N_DATES), control_matrix[:, :largest_k]])
coefficient_variance = np.diag(np.linalg.inv(design.T @ design))[1:]

print(f"K = {largest_k}, posterior sigma = {sigma_hat:.4f}")
print(
    f"  mean variance inflation factor:          {(coefficient_variance * N_DATES).mean():.2f}"
)
print(f"  likelihood-only sd, orthogonal columns:  {sigma_hat / np.sqrt(N_DATES):.4f}")
print(
    "  likelihood-only sd, these columns:      "
    f"{(sigma_hat * np.sqrt(coefficient_variance)).mean():.4f}"
)
for arm, gamma_sd in at_largest_k["gamma_sd"].items():
    print(f"  observed posterior sd, {arm:18s}{gamma_sd:.4f}")

### The seasonal block is a control block too

`yearly_seasonality=2` is an unremarkable thing to write, and it adds four covariates to the model: `sin_1`, `sin_2`, `cos_1`, `cos_2`. Exactly one of them is real. The DGP's seasonality *is* `sin_1`, so the other three have a true coefficient of exactly zero — they are the same phantom columns as the tail of the control block, arriving through a keyword argument rather than a dataframe.

That makes them a clean test of the notebook's claim, because we know the answer in advance. `gamma_sin_1` should land near its true value and the other three should be pinned near zero, and how close each arm gets to that is a property of the prior rather than of the data.

The two panels below track exactly those two quantities against $K$. The right-hand one is the more interesting: under the library default the phantom coefficients are held only by `Laplace(b=1)`, which on a max-scaled target is enormous, so the likelihood decides how much of the residual they mop up. Under Split R2D2 they draw from the shared Dirichlet, so their allowance shrinks as controls arrive — the flat Dirichlet gives the Fourier block $4/(K + 4)$ of the budget in expectation, from all of it at $K = 0$ down to about a seventh at $K = 24$.

There is a caveat worth stating in the same breath, and it is the one place this setup treats the two blocks unequally. R2D2 shares one budget across coefficients as though every covariate had unit variance, and our controls are standardised so that they do. The Fourier basis is not: each node has standard deviation $1/\sqrt{2} \approx 0.707$, so a Fourier coefficient needs to be $\sqrt{2}$ larger than a control's to buy the same variance. The seasonal block is therefore very slightly under-allowed relative to the controls. It is a factor of $\sqrt 2$ on one block, not a factor in $K$, so it does not touch the dimension argument — but it is the same footnote-7 issue as before, and it is why the caveat about standardising covariates matters more once seasonality joins the budget.

In [ ]:
true_sin_1 = truth["gamma_fourier"][FOURIER_NODES.index("sin_1")]

fig, axes = plt.subplots(ncols=2, figsize=(13, 5), sharex=True, layout="constrained")

for arm, color in ARM_COLORS.items():
    subset = by_arm(fits, arm)
    axes[0].plot(subset["K"], subset["gamma_sin_1"], marker="o", color=color, label=arm)
    axes[1].plot(subset["K"], subset["phantom_rms"], marker="o", color=color, label=arm)

axes[0].axhline(true_sin_1, color="black", linestyle="--", linewidth=2, label="truth")
axes[0].set(
    xlabel="number of controls K",
    ylabel=r"posterior mean $\gamma_{\mathrm{sin}_1}$",
    title="The one real seasonal coefficient",
)
axes[0].legend(fontsize=9)

axes[1].axhline(0.0, color="black", linestyle="--", linewidth=2, label="truth (zero)")
axes[1].set(
    xlabel="number of controls K",
    ylabel="posterior RMS",
    title=f"The {len(PHANTOM_NODES)} phantom seasonal coefficients: {', '.join(PHANTOM_NODES)}",
)
axes[1].legend(fontsize=9)

fig.suptitle(
    "`yearly_seasonality=2` adds one real covariate and three that do nothing",
    fontsize=14,
)
plt.show()

A more compact view of the headline result: interval length, and the posterior median against the truth.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(13, 5), sharex=True, layout="constrained")

for arm, color in ARM_COLORS.items():
    subset = by_arm(roas_table, arm)
    axes[0].plot(
        subset["K"],
        subset["interval_length"],
        marker="o",
        color=color,
        label=arm,
    )
    axes[1].plot(
        subset["K"],
        subset["median"],
        marker="o",
        color=color,
        label=arm,
    )
    axes[1].fill_between(
        subset["K"], subset["q05"], subset["q95"], color=color, alpha=0.12
    )

axes[0].set(
    xlabel="number of controls K",
    ylabel="90% interval length",
    title="Posterior interval length for total ROAS",
)
axes[0].legend(fontsize=9)

axes[1].axhline(
    truth["total_roas"], color="black", linestyle="--", linewidth=2, label="truth"
)
axes[1].set(
    xlabel="number of controls K",
    ylabel="total ROAS",
    title="Posterior median and 90% interval",
)
axes[1].legend(fontsize=9)

plt.show()

### Supporting view: is the variance going to the right place?

Total ROAS is a summary; it helps to see how each fit splits the target's variation between media, controls, seasonality and residual, and to compare that against the truth. Because the target is max-scaled to a maximum of 1, the model's internal scale is exactly 1, so posterior and true quantities are directly comparable.

In [ ]:
fits[
    ["K", "arm", "media_share", "control_share", "seasonal_share", "posterior_r2"]
].round(4)

In [ ]:
# Match variance_shares(): Var(media) over Var(media) + Var(control) +
# Var(seasonality) + sigma^2.
true_media_var_share = truth["media_var_share"]

fig, axes = plt.subplots(ncols=2, figsize=(13, 5), sharex=True, layout="constrained")

for arm, color in ARM_COLORS.items():
    subset = by_arm(fits, arm)
    axes[0].plot(
        subset["K"],
        subset["media_share"],
        marker="o",
        color=color,
        label=arm,
    )
    axes[1].plot(
        subset["K"],
        subset["posterior_r2"],
        marker="o",
        color=color,
        label=arm,
    )

axes[0].axhline(
    true_media_var_share,
    color="black",
    linestyle="--",
    linewidth=2,
    label="truth",
)
axes[0].set(
    xlabel="number of controls K",
    ylabel="media share of target variance",
    title="Posterior media variance share",
)
axes[0].legend(fontsize=9)

axes[1].axhline(truth["r2"], color="black", linestyle="--", linewidth=2, label="truth")
axes[1].set(
    xlabel="number of controls K",
    ylabel="posterior $R^2$",
    title=r"Posterior $R^2$: $\mathrm{Var}(\mu) / (\mathrm{Var}(\mu) + \sigma^2)$",
)
axes[1].legend(fontsize=9)

plt.show()

<!-- STALE-NUMBERS: every XX below reads off the executed outputs; refresh after re-running. -->
Both panels rise with $K$, and that is expected: the controls are real signal, so a model that includes more of them should explain more. The question is where each arm stops, and this is where the independent priors have historically come out looking better.

In the right panel they land on a posterior $R^2$ of XX and XX at $K = 24$ against a true 0.770, while R2D2 gives XX. Residual scale tells the same story: XX and XX against a true 0.0583, where R2D2 gives XX. If R2D2 is again mildly *pessimistic* about the fit, that is worth stating plainly rather than hiding, because it is what shrinkage costs and it is the trade the prior is making: a slightly worse description of the target's variance in exchange for a better estimate of media's level.

The left panel is the negative result from the previous section, drawn: XX, XX and XX against a true media variance share of 0.150.

### Diagnostics

Sampling quality per fit. This matters for interpretation: if a fit that produces a badly placed ROAS posterior also shows no sampling trouble, then nothing would have warned a practitioner that something was wrong. The flag below uses the conventional $\hat{R} \le 1.01$ threshold.

One asymmetry to look for, because the previous run of this experiment showed it: it was the Split R2D2 fits — the *more accurate* ones — that picked up divergences at large $K$, not the independent ones. Deriving both the coefficient budget and the likelihood's residual scale from the same `r2`/`total_sigma` pair creates a funnel that gets harder to sample as more coefficients pull on `total_sigma`. If that repeats here, the two diagnostics agree with each other and not with what actually matters: both single out the more accurate fit for suspicion, while the arm with the larger ROAS error samples cleanly.

In [ ]:
fits.assign(
    r_hat_flag=lambda frame: np.where(frame["max_r_hat"] > 1.01, "check", ""),
).round(4)

### Repetition study

A single dataset cannot separate a real effect from one draw's luck, and the paper's own effect is modest — Table 4 reports RMSE 0.30 against 0.27 and coverage 0.87 against 0.92 over many replications. The function below is the Table 4 analogue: repeat the whole pipeline over `n_reps` datasets and report 90% interval length, coverage and RMSE of total ROAS per $K$ and arm.

The first replication reuses the dataset above, so `n_reps` controls how many *additional* datasets get simulated and fit; it is called with 5 below, which is 60 fits in total. Dropping it to 1 makes the notebook much faster at the cost of turning "coverage" into a single indicator per cell rather than a rate.

In [ ]:
def run_experiment(n_reps: int = 1, base_seed: int = SEED) -> pd.DataFrame:
    """Repeat simulate + fit over `n_reps` datasets."""
    records = []
    for rep in range(n_reps):
        seed = base_seed + rep
        if rep == 0:
            rep_data, rep_truth = data, truth
        else:
            rep_data, rep_truth = simulate(seed)

        for k in K_GRID:
            for arm in ARM_NAMES:
                if rep == 0:
                    draws = roas_draws[k, arm]
                else:
                    draws, _ = fit_for_k(rep_data, k, arm, seed=seed)

                q05, q95 = np.quantile(draws, [0.05, 0.95])
                records.append(
                    {
                        "rep": rep,
                        "K": k,
                        "arm": arm,
                        "interval_length": q95 - q05,
                        "covers": bool(q05 <= rep_truth["total_roas"] <= q95),
                        "squared_error": (draws.mean() - rep_truth["total_roas"]) ** 2,
                    }
                )
    return pd.DataFrame(records)


reps = run_experiment(n_reps=5)

reps.groupby(["K", "arm"]).agg(
    interval_length=("interval_length", "mean"),
    coverage=("covers", "mean"),
    rmse=("squared_error", lambda errors: np.sqrt(errors.mean())),
).round(4)

## Conclusion

<!-- STALE-NUMBERS: every XX below reads off the executed outputs; refresh after re-running. -->
Adding a covariate to an MMM is not free, and the cost is not paid in the place you would look for it. The covariates here are neutral by construction — independent of spend, incapable of confounding ROAS — and the data never change across fits. Everything we measured came from the priors on `gamma_control` and `gamma_fourier`.

What the experiment shows:

- **The mechanism is visible before any MCMC.** Under independent priors with fixed scales, the implied prior $R^2$ rises monotonically towards 1 as covariates are added, and the share of prior variance available to media collapses. Under the library defaults this has already happened by $K = XX$. R2D2 is flat in $K$, because the Dirichlet splits a fixed budget rather than accumulating one.
- **It reaches the reported number, at ordinary dimensions.** At 24 controls plus 4 Fourier nodes against $n = 130$ — $p/n$ of 0.22, nothing exotic — the absolute ROAS error is XX under the library defaults and XX hand-tightened, against XX under R2D2 and a true ROAS of 2.008.
- **Scale is not the fix; dimension-awareness is.** The two independent arms differ by a factor of 20 in coefficient scale yet land within XX of each other on ROAS. Tightening a prior you believe is too wide does not address the problem, because the problem is that the total budget grows with the number of coefficients.
- **Seasonality is part of the problem, not exempt from it.** `yearly_seasonality=2` adds four covariates of which three have a true coefficient of zero. Under the library default `Laplace(b=1)` those three carry a posterior RMS of XX at $K = 0$; under a budget shared with the controls, XX. Whatever your seasonality is doing for you, its phantom nodes are competing for the same explained variance your media coefficients need.
- **The failure is in the level, not the variance.** The three arms agree on media's share of target variance to within XX and all overshoot the truth. They disagree on media's share of mean *sales* — XX, XX and XX against a true 0.151 — and that is the quantity ROAS is made of.

Practical takeaways:

1. **The default priors do not know how many coefficients you have.** `Prior("Normal", mu=0, sigma=2, dims="control")` and `Prior("Laplace", mu=0, b=1, dims="fourier_mode")` are strong statements about total explained variance, and they get stronger every time you add a column or raise `yearly_seasonality`.
2. **Check the implied prior $R^2$.** It costs one `sample_prior_predictive` call and no sampling, and it is the fastest way to find out whether your priors are quietly asserting a near-perfect fit.
3. **Ask what your covariates could mimic.** Regress your media contribution — fitted, if you have no truth — on the control block and on the Fourier basis, and look at the $R^2$. It is a two-line diagnostic, it tells you how much capacity those blocks have to explain media away, and it is far more informative than the pairwise correlations most people check.
4. **Standardise your controls.** All three arms are stated on the scale of the coefficients, and `MMM` does not scale controls for you. A shared variance budget is only meaningful once the columns are comparable — note that the Fourier basis is not standardised either, at $1/\sqrt2$ per node.
5. **Put seasonality in the budget with the controls.** `dims={"control": "control", "fourier": "fourier_mode"}` is one line, and it is the more usual case: nearly every MMM has yearly seasonality, and most of its Fourier nodes are doing nothing in particular.
6. **A split prior is enough.** Putting R2D2 on the controls and the seasonal block leaves every media prior untouched, so the media parameterisation you have tuned stays exactly as it was. The joint version, which shrinks media too, is a different and — per the paper — considerably worse idea.
7. **Diagnostics can point the wrong way.** `pymc_marketing.r2d2.R2D2` derives the residual scale from the same `r2`/`total_sigma` pair that sets the coefficient budget, which is a harder funnel than treating the residual scale as its own free parameter — so the R2D2 fits are the ones liable to report divergences and elevated $\hat R$ even when they are the more accurate fits. A practitioner reading only sampler diagnostics could flag the fit to trust and wave through the ones that are misleading.

### What this notebook does not do

- **A proper repetition study.** `run_experiment` is called with five replications, which is enough to show a direction and not enough to size an effect. That needs `n_reps` in the tens and a compute budget to match. This is the most important gap.
- **Confounded controls.** Spend here is generated without reference to the controls or the calendar. Real marketers spend into high season, which makes seasonality a genuine confounder and changes the calculus entirely — dropping it then biases ROAS for reasons that have nothing to do with prior geometry.
- **Higher-order seasonality.** We stop at `yearly_seasonality=2`, so three of four nodes are phantoms. At order 4 or 6 — both common — the ratio gets worse, and the shared budget has more cells to divide. Nothing here measures where that stops being benign.
- **The joint R2D2 prior**, spanning media and control coefficients together, which the paper shows is badly biased.
- **Multiple geos.** The class used here is the dims-based `MMM` either way, so adding `dims=("geo",)` is mostly a matter of widening the prior dims and the spend generator.

## References

- [*To select or not to select: that is the question*](https://arxiv.org/abs/2606.22850) — the source of the experiment this notebook translates. Experiment 4, Section 5.4, Figures 4 and 9, Table 4.
- Zhang, Y. D., Naughton, B. P., Bondell, H. D., & Reich, B. J. (2022). [Bayesian Regression Using a Prior on the Model Fit: The R2-D2 Shrinkage Prior](https://doi.org/10.1080/01621459.2020.1825449). *Journal of the American Statistical Association*.
- Cinelli, C., Forney, A., & Pearl, J. (2024). [A Crash Course in Good and Bad Controls](https://doi.org/10.1177/00491241221099552). *Sociological Methods & Research*.
- Jin, Y., Wang, Y., Sun, Y., Chan, D., & Koehler, J. (2017). [Bayesian Methods for Media Mix Modeling with Carryover and Shape Effects](https://research.google/pubs/pub46001/).

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,pytensor,pymc